In [17]:
import pandas as pd
import numpy as np
import json
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [18]:
# Load training and test data
X_train = pd.read_csv('data/X_train.csv').values
y_train = pd.read_csv('data/y_train.csv').values.ravel()
X_test = pd.read_csv('data/X_test.csv').values
y_test = pd.read_csv('data/y_test.csv').values.ravel()

# Load metadata for genre names
with open('data/metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of genres: {metadata['n_genres']}")

Training samples: 37588
Test samples: 9397
Number of features: 14
Number of genres: 24


## Hyperparameter Tuning

In [19]:
# Define hyperparameter grid
hyperparameter_grid = [
    # Euclidean - uniform weights
    {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 11, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 15, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 19, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 25, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2},
    
    # Euclidean - distance weights
    {'n_neighbors': 5, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 7, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 9, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 11, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 15, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    {'n_neighbors': 19, 'weights': 'distance', 'metric': 'euclidean', 'p': 2},
    
    # Manhattan - uniform weights
    {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 11, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 15, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 19, 'weights': 'uniform', 'metric': 'manhattan', 'p': 1},
    
    # Manhattan - distance weights
    {'n_neighbors': 5, 'weights': 'distance', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 7, 'weights': 'distance', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 11, 'weights': 'distance', 'metric': 'manhattan', 'p': 1},
    {'n_neighbors': 15, 'weights': 'distance', 'metric': 'manhattan', 'p': 1},
]

print(f"Testing {len(hyperparameter_grid)} different hyperparameter combinations...")

Testing 25 different hyperparameter combinations...


## Train and Evaluate Models

In [20]:
trials = []
best_accuracy = 0
best_hyperparameters = None
best_confusion_matrix = None
total_train_time = 0
total_test_time = 0

# Train and evaluate each configuration
for i, params in enumerate(hyperparameter_grid, 1):
    print(f"\nTrial {i}/{len(hyperparameter_grid)}: {params}")
    
    # Create and train model
    train_start = time.time()
    model = KNeighborsClassifier(
        n_neighbors=params['n_neighbors'],
        weights=params['weights'],
        metric=params['metric'],
        p=params['p']
    )
    model.fit(X_train, y_train)
    train_time = time.time() - train_start
    total_train_time += train_time
    
    # Make predictions
    test_start = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - test_start
    total_test_time += test_time
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Train time: {train_time:.2f}s, Test time: {test_time:.2f}s")
    
    # Store trial results
    trials.append({
        'hyperparameters': params.copy(),
        'confusion_matrix': cm.tolist()
    })
    
    # Update best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_hyperparameters = params.copy()
        best_confusion_matrix = cm.tolist()
        print(f"✓ New best accuracy: {accuracy:.4f}")

print(f"\n{'='*60}")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best Hyperparameters: {best_hyperparameters}")
print(f"Total Training Time: {total_train_time:.2f}s")
print(f"Total Testing Time: {total_test_time:.2f}s")
print(f"{'='*60}")


Trial 1/25: {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.3627
Train time: 0.02s, Test time: 1.41s
✓ New best accuracy: 0.3627

Trial 2/25: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.3627
Train time: 0.02s, Test time: 1.41s
✓ New best accuracy: 0.3627

Trial 2/25: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.3960
Train time: 0.01s, Test time: 1.65s
✓ New best accuracy: 0.3960

Trial 3/25: {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.3960
Train time: 0.01s, Test time: 1.65s
✓ New best accuracy: 0.3960

Trial 3/25: {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.4054
Train time: 0.01s, Test time: 1.68s
✓ New best accuracy: 0.4054

Trial 4/25: {'n_neighbors': 9, 'weights': 'uniform', 'metric': 'euclidean', 'p': 2}
Accuracy: 0.4054
Train time: 0.01s, Test time: 1.68s
✓ New best accuracy: 0.4054

Tri

## Save Results to JSON

In [21]:
# Prepare JSON
output = {
    'model_name': 'K-Nearest Neighbors',
    'person_name': 'Austin Bell',
    'best_hyperparameters': best_hyperparameters,
    'best_confusion_matrix': best_confusion_matrix,
    'trials': trials,
    'total_train_time': round(total_train_time, 2),
    'total_test_time': round(total_test_time, 2)
}

# Save to file
output_path = 'output/k_nearest_neighbors.json'
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {output_path}")
print(f"\nFinal Summary:")
print(f"Model: K-Nearest Neighbors")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best k: {best_hyperparameters['n_neighbors']}")
print(f"Best weights: {best_hyperparameters['weights']}")
print(f"Best metric: {best_hyperparameters['metric']}")

Results saved to output/k_nearest_neighbors.json

Final Summary:
Model: K-Nearest Neighbors
Best Accuracy: 0.4429
Best k: 19
Best weights: uniform
Best metric: manhattan
